In [1]:
import os, json, pathlib, re, ast
from openai import OpenAI 
import openai
from tqdm import tqdm
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from typing import List, Dict, Optional
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_openai import OpenAIEmbeddings


In [2]:
# Load API
load_dotenv()
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
OLLAMA_BASE = "http://localhost:11434/v1" 
MODEL_NAME = "llama3:8b-instruct-q4_K_M" 
TEMPERATURE = 1.0

ollama_client = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")
print("Ollama client ready →", OLLAMA_BASE, "model:", MODEL_NAME)


# Index file
INDEX_DIR = pathlib.Path("faiss_disaster_information_index")

# Load files
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_llama.csv"
df = pd.read_csv(src_file)
# df = df.head(200)

Ollama client ready → http://localhost:11434/v1 model: llama3:8b-instruct-q4_K_M


# FAISS Vectorstore

In [4]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [5]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [6]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [7]:
EXAMPLES = pathlib.Path("../data/Examples.txt").read_text(encoding="utf-8")

In [8]:
disaster_docs = []
groups = DISASTER_CLASSIFICATION["Disaster Group"]
for group_name, meta in groups.items():
        desc  = meta.get("Description", "")
        types = meta.get("Disaster Types", {})

        block = f"### Disaster Group: {group_name}\n{desc}\n\nDisaster Types:\n"
        for t_name, t_desc in types.items():
            block += f"- {t_name}: {t_desc}\n"

        doc = Document(
            page_content = block.strip(),
            metadata     = {"group": group_name, "level": "G1"}
        )
        disaster_docs.append(doc)

In [9]:
magnitude_docs = []
for dtype, info in MAGNITUDE_INFORMATION.items():
    block = (
        f"### Magnitude Guidance\n"
        f"Disaster Type: {dtype}\n"
        f"Property      : {info['property']}\n"
        f"Unit          : {info['unit']}"
    )

    doc = Document(
        page_content = block,
        metadata     = {"type" : dtype, "level": "M1" }
    )
    magnitude_docs.append(doc)

In [10]:
fields_docs = []
for section_name, fields in FIELD_SCHEMA.items():
    for fname, meta in fields.items():
        block = (
            f"### Field: {fname}\n"
            f"Section      : {section_name}\n"
            f"Type         : {meta['Type']}\n"
            f"Description  : {meta.get('Description', meta.get('Rescription',''))}"
        )

        doc = Document(
            page_content = block,
            metadata = {"section": section_name, "field"  : fname, "level"  : "F0"
                }
        )
        fields_docs.append(doc)

In [11]:
examples_docs = []
block = re.split(r"Example\s+\d+:\s*", EXAMPLES)[1:]

for idx, chunk in enumerate(block, start=1):
    doc = Document(
        page_content = chunk.strip(),
        metadata     = {"example_id": idx, "level": "E1"}
    )
    examples_docs.append(doc)

In [12]:
for k in ("OPENAI_BASE_URL", "OPENAI_API_BASE"):
    os.environ.pop(k, None)

EMB_MODEL = "text-embedding-3-small"
emb = OpenAIEmbeddings(model=EMB_MODEL, api_key=OPENAI_KEY)

if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(
        str(INDEX_DIR),
        embeddings=emb,  
        allow_dangerous_deserialization=True
    )
else:
    print("Building FAISS index from scratch … ")

    vectorstore = FAISS.from_documents(disaster_docs, emb) 
    vectorstore.add_documents(magnitude_docs)
    vectorstore.add_documents(fields_docs)
    vectorstore.add_documents(examples_docs)
    vectorstore.save_local(str(INDEX_DIR))

print(f"✅  Vectorstore ready, ntotal = {vectorstore.index.ntotal}")

✅  Vectorstore ready, ntotal = 41


# Prompt

In [14]:
OUTPUT_FIELDS = {
  "Event": "fields name",
  "Geographical Information": "fields name",
  "Effect": "fields name"
}

In [15]:
OUTPUT_VALUES = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [16]:
def restructure_example(doc: Document):
    
    txt = doc.page_content

    # description
    e_desc = re.search(r"Description:\s*(.*?)\s*Fields output:", txt, re.S)
    if not e_desc:
        raise ValueError(f"Did not find description, example_id={doc.metadata.get('example_id')}")
    description = e_desc.group(1).strip()

    # fields / values
    e_fields = re.search(r"Fields output:\s*(\{.*?\})\s*Values output:", txt, re.S)
    e_values = re.search(r"Values output:\s*(\{.*\})\s*$", txt, re.S)
    
    if not (e_fields and e_values):
        raise ValueError(f"Did not find description fields/values, example_id={doc.metadata.get('example_id')}")

    example_fields  = ast.literal_eval(e_fields.group(1).strip())
    example_values  = ast.literal_eval(e_values.group(1).strip())

    # restructure
    stage1 = {
        "description":    description,
        "output_fields":  example_fields,
    }
    stage2 = {
        "description":    description,
        "output_values":  example_values,
    }
    return stage1, stage2


In [17]:
def fields_extraction_prompt (example1: Document, example2: Document) -> str:
    
    stage1_dict1, _ = restructure_example(example1)
    stage1_dict2, _ = restructure_example(example2)
    
    prompt = f"""
        Please refer FIELD_SCHEMA first, then list every field that is
        mentioned (even once) or is marked as 'Required' in DISASTER_DESCRIPTION. 
        Output must contain field names only.
        Finally return a single JSON object that conforms precisely to OUTPUT_FIELDS.
        Below there are two reference EXAMPLE1_STAGE1 and EXAMPLE2_STAGE1 
        for one disaster's description and its output fields.
        Return ONLY one valid minified JSON object.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        OUTPUT_FIELDS:
        {json.dumps(OUTPUT_FIELDS, ensure_ascii=False)}

        EXAMPLE_STAGE1:
        {json.dumps(stage1_dict1, ensure_ascii=False, indent=4)}

        EXAMPLE2_STAGE1:
        {json.dumps(stage1_dict2, ensure_ascii=False, indent=4)}

    """
    return prompt

In [18]:
def values_extraction_prompt(
    fields_docs: List[Document], 
    example: Document,
    categorical_values: Optional[Document]
) -> str:
    
    PROMPT = """
    Please extract all information regarding only the fields in EXTRACTED_FIELDS 
    from text DISASTER_DESCRIPTION (refering the information in FIELD_SCHEMA and using only 
    the categorical options in CATEGORICAL_VALUES). 
    Finally return a single JSON object that conforms precisely to OUTPUT_VALUES.
    Below there is a reference EXAMPLE_STAGE2 for one disaster's description and its output values.
    Return ONLY one valid minified JSON object.
"""

    parts = ["FIELD_SCHEMA:"]
    parts += [d.page_content for d in fields_docs]

    parts += ["CATEGORICAL_VALUES:"]
    parts += [d.page_content for d in categorical_values] 

    parts += [
        "OUTPUT_VALUES:",
        json.dumps(OUTPUT_VALUES, ensure_ascii=False)
    ]

    _, stage2_dict= restructure_example(example)
    parts += ["EXAMPLE_STAGE2:",
              json.dumps(stage2_dict, ensure_ascii=False, indent=4)]
    
    return PROMPT + "\n\n" + "\n\n".join(parts)


# Implement

In [20]:
def _brace_scan(s: str):
    if not s:
        return None
    stack, start = [], None
    for i, ch in enumerate(s):
        if ch in "{[":
            if not stack:
                start = i
            stack.append(ch)
        elif ch in "}]":
            if not stack:
                continue
            left = stack.pop()
            if (left == "{" and ch == "}") or (left == "[" and ch == "]"):
                if not stack and start is not None:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        pass
    return None

In [21]:
def extract_json_from_text(text: str):
    if not text:
        return None
    text = text.strip()

    # 1) JSON Format
    if text and text[0] in "[{":
        try:
            return json.loads(text)
        except Exception:
            pass

    # 2) ```json ... ```
    m = re.search(r"```(?:json|JSON)?\s*([\s\S]*?)\s*```", text)
    if m:
        candidate = m.group(1).strip()
        try:
            return json.loads(candidate)
        except Exception:
            parsed = _brace_scan(candidate)
            if parsed is not None:
                return parsed

    # 3) Find the first {} or []
    parsed = _brace_scan(text)
    if parsed is not None:
        return parsed

    return None

In [22]:
def call_llm(prompt_text: str, description_text: str, extracted_fields_text=None) -> dict:
    if extracted_fields_text is None:
        user_content = description_text or ""
    else:
        user_content = json.dumps({
            "DISASTER_DESCRIPTION": description_text,
            "EXTRACTED_FIELDS": extracted_fields_text
        }, ensure_ascii=False)

    def _ask(sys_prompt: str, user_msg: str):
        return ollama_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user",   "content": user_msg}
            ],
            temperature=1.0, 
            extra_body={
                "format": "json",
                "options": {"num_ctx": 16384, "num_predict": 768}
            }
        )

    last_head = ""

    try:
        resp = _ask(prompt_text, user_content)
        text = (resp.choices[0].message.content or "").strip()
        parsed = extract_json_from_text(text)
        if parsed is not None:
            return parsed
        last_head = repr(text[:160])
    except Exception as e:
        print("[GEN] error:", e)

    for i in range(1, 3):
        try:
            resp2 = _ask(prompt_text, user_content)
            text2 = (resp2.choices[0].message.content or "").strip()
            parsed2 = extract_json_from_text(text2)
            if parsed2 is not None:
                return parsed2
            last_head = repr(text2[:160])
            print(f"[WARN] Regen attempt {i} still not JSON. Head: {last_head}")
        except Exception as e:
            print(f"[GEN] error on regen {i}:", e)

    print("[WARN] Still not valid JSON after regenerations. Head:", last_head)
    return None 


In [23]:
def process_record(description: str) -> dict:
    if not isinstance(description, str):
        return None 

    # examples
    e1_docs = vectorstore.similarity_search(
        description, 
        k=2,
        filter={"level": "E1"}
    )
    if len(e1_docs) < 2:
        used_ids = {d.metadata.get("example_id") for d in e1_docs}
        candidates = [d for d in examples_docs if d.metadata.get("example_id") not in used_ids]
        while len(e1_docs) < 2 and candidates:
            e1_docs.append(candidates.pop())
    
    # Stage1: fields
    p1 = fields_extraction_prompt(e1_docs[0], e1_docs[1])
    r1 = call_llm(p1, description)

    if not isinstance(r1, dict) or r1 is None:
        return None
    
    # Stage2: values
    # 2-1: fields_docs
    if not isinstance(r1, dict):
        return None

    field_list = []
    ok = True
    for v in r1.values():
        if not isinstance(v, list) or not all(isinstance(x, str) for x in v):
            ok = False
            break
        field_list.extend(v)

    if not ok or not field_list:
        return None
   
    
    f0_docs = []
    for fname in field_list:
        doc = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "F0", "field": fname}
        )
        if doc:
            f0_docs.append(doc[0])
        else:
            # fallback
            field_doc_map = {d.metadata["field"]: d.page_content for d in fields_docs}
            f0_docs.append(
                Document(
                    page_content=field_doc_map.get(fname, ""),
                    metadata={"level": "F0", "field": fname}
                )
            )
    
    # 2-2: categorical values
    g1_docs = vectorstore.similarity_search(
        description,
        k=2,
        filter={"level": "G1"}
    )
    
    m1_doc = None
    if {"Magnitude", "Magnitude Scale"} & set(field_list):
        hits = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "M1"}
        )
        m1_doc = hits[0] if hits else None

    categorical_values = g1_docs + ([m1_doc] if m1_doc else [])


    
    # 2-3: extraction
    p2 = values_extraction_prompt(
        fields_docs      = f0_docs,
        example = e1_docs[0],
        categorical_values = categorical_values
    )
    r2 = call_llm(p2, description, r1)
    
    if r2 is None:
        return None
    
    return {
        "llama_fields": json.dumps(r1, ensure_ascii=False),
        "llama_values": json.dumps(r2, ensure_ascii=False)
    }


In [24]:
results = []
for desc in tqdm(df["description"], desc="Processing"):
    out = process_record(desc)
    if out is None:
        results.append({"llama_fields": None, "llama_values": None})
    else:
        results.append(out)

df_out = pd.concat(
    [df.reset_index(drop=True), pd.DataFrame(results)],
    axis=1
)

df_out.to_csv(out_file, index=False)
print(f"Output saved to {out_file}")


Processing:   0%|                           | 6/1588 [03:15<16:53:57, 38.46s/it]

[WARN] Regen attempt 1 still not JSON. Head: 'Here is the extracted information in a single JSON object that conforms to the OUTPUT_VALUES schema:\n\n```\n{\n    "output_values": {\n        "Event": {\n          '


Processing:   1%|▏                         | 12/1588 [07:29<17:43:12, 40.48s/it]

[WARN] Regen attempt 1 still not JSON. Head: 'From the given description, the following fields are mentioned:\n\n- Disaster Group\n- Disaster Type\n- Disaster Date (YMD)\n- Cause\n- Country\n- Geographical Level 0'


Processing:   1%|▎                         | 18/1588 [10:52<13:08:02, 30.12s/it]

...

[WARN] Regen attempt 1 still not JSON. Head: 'Here are the fields mentioned or required in these examples:\n\n1. EVENT:\n    - Disaster Group\n    - Disaster Type\n    - Disaster Date (YMD)\n\n2. GEOGRAPHICAL INFO'


Processing: 100%|███████████████████████▉| 1583/1588 [14:07:56<03:05, 37.02s/it]

[WARN] Regen attempt 1 still not JSON. Head: "According to the FIELD_SCHEMA, the fields mentioned or marked as 'Required' in the provided disaster description are:\n\n- Event: Disaster Group, Disaster Type, D"


Processing: 100%|████████████████████████| 1588/1588 [14:11:04<00:00, 32.16s/it]

Output saved to extracted_llama.csv
